# Task 2: Predict Future Stock Prices (Short-Term)
## DevelopersHub Corporation — AI/ML Engineering Internship

---

### Problem Statement
Use historical stock market data to **predict the next day's closing price** using machine learning regression models.

### Goal
- Fetch real stock data using the `yfinance` API
- Engineer meaningful features from OHLCV data
- Train and compare **Linear Regression** and **Random Forest** models
- Visualize actual vs predicted prices
- Evaluate using standard regression metrics

### Stock Selected
**Apple Inc. (AAPL)** — one of the most liquid and widely-traded stocks

---

## 1. Setup & Dependencies

In [ ]:
# Install required libraries (run once)
!pip install yfinance pandas numpy scikit-learn matplotlib seaborn --quiet
print("✅ Libraries installed.")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Plot styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
COLORS = {'actual': '#2196F3', 'linear': '#FF5722', 'rf': '#4CAF50', 'accent': '#9C27B0'}

print("✅ All imports successful.")

## 2. Fetch Stock Data via yfinance API

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
TICKER   = "AAPL"        # Stock symbol (try: TSLA, MSFT, GOOGL)
PERIOD   = "2y"          # 2 years of daily data
INTERVAL = "1d"          # Daily candles
# ──────────────────────────────────────────────────────────────────────────────

print(f"📡 Fetching {TICKER} data from Yahoo Finance...")
stock = yf.Ticker(TICKER)
df = stock.history(period=PERIOD, interval=INTERVAL)

# Drop timezone info for cleaner display
df.index = df.index.tz_localize(None)

print(f"✅ Fetched {len(df)} trading days of data")
print(f"   Date range: {df.index[0].date()} → {df.index[-1].date()}")
print(f"   Columns: {list(df.columns)}")

df.head()

In [ ]:
# Shape, dtypes, and basic stats
print(f"Shape: {df.shape}")
print(f"\nColumn Info:")
print(df.info())

print(f"\nDescriptive Statistics:")
df[['Open','High','Low','Close','Volume']].describe().round(2)

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Drop any NaN rows (rare but possible for stock splits/halts)
df.dropna(inplace=True)
print(f"\n✅ Dataset is clean. {len(df)} rows ready for modelling.")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Price History Chart ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

# Closing price
axes[0].plot(df.index, df['Close'], color=COLORS['actual'], linewidth=1.5, label='Close Price')
axes[0].fill_between(df.index, df['Close'], df['Close'].min(), alpha=0.1, color=COLORS['actual'])
axes[0].set_title(f'{TICKER} — Closing Price (Last 2 Years)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price (USD)')
axes[0].legend()

# Volume
axes[1].bar(df.index, df['Volume'], color=COLORS['accent'], alpha=0.6, width=1)
axes[1].set_title('Daily Trading Volume', fontsize=12)
axes[1].set_ylabel('Volume')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

plt.tight_layout()
plt.savefig('eda_price_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Distributions ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, color in zip(axes, ['Open', 'Close', 'Volume'],
                          [COLORS['actual'], COLORS['linear'], COLORS['accent']]):
    ax.hist(df[col], bins=40, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.1f}')
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.set_xlabel(col)
    ax.legend(fontsize=9)

plt.suptitle(f'{TICKER} Feature Distributions', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation Heatmap ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

corr = df[['Open', 'High', 'Low', 'Close', 'Volume']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=ax, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title(f'{TICKER} — Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nInsight: Open, High, Low, Close are strongly correlated (>0.99).")
print("Volume has a weaker/negative correlation with price — useful as a distinguishing feature.")

## 4. Feature Engineering

Raw OHLCV columns alone make poor features — the model would just be learning that *today's close ≈ yesterday's close*. We engineer richer, forward-looking features:

| Feature | Meaning |
|---|---|
| `Open`, `High`, `Low`, `Volume` | Same-day inputs (available before close) |
| `Price_Range` | High − Low — measures intraday volatility |
| `Open_Close_Gap` | Today's Open − Yesterday's Close — gap-up/gap-down signal |
| `MA_5`, `MA_10` | 5-day and 10-day moving averages of Close |
| `Vol_MA_5` | 5-day moving average of Volume |
| `Lag_1`, `Lag_2`, `Lag_3` | Previous 1/2/3 days' closing prices |
| `Pct_Change` | Day-over-day % change in close |

In [ ]:
data = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# ── Technical features ────────────────────────────────────────────────────────
data['Price_Range']    = data['High'] - data['Low']
data['Open_Close_Gap'] = data['Open'] - data['Close'].shift(1)
data['MA_5']           = data['Close'].rolling(5).mean()
data['MA_10']          = data['Close'].rolling(10).mean()
data['Vol_MA_5']       = data['Volume'].rolling(5).mean()
data['Lag_1']          = data['Close'].shift(1)
data['Lag_2']          = data['Close'].shift(2)
data['Lag_3']          = data['Close'].shift(3)
data['Pct_Change']     = data['Close'].pct_change() * 100

# ── Target: NEXT day's closing price ─────────────────────────────────────────
data['Target'] = data['Close'].shift(-1)

# Drop rows with NaNs (created by rolling windows and shifts)
data.dropna(inplace=True)

print(f"✅ Feature engineering complete.")
print(f"   Dataset shape: {data.shape}")
print(f"\nFeature columns:")
feature_cols = [c for c in data.columns if c != 'Target']
print(feature_cols)

data.head(3)

## 5. Train / Test Split

> **Important:** For time-series data we must **not** shuffle. We use the last 20% of chronological data as the test set to simulate real-world forward prediction.

In [ ]:
FEATURE_COLS = [
    'Open', 'High', 'Low', 'Volume',
    'Price_Range', 'Open_Close_Gap',
    'MA_5', 'MA_10', 'Vol_MA_5',
    'Lag_1', 'Lag_2', 'Lag_3', 'Pct_Change'
]

X = data[FEATURE_COLS].values
y = data['Target'].values
dates = data.index

# Chronological 80/20 split — NO shuffling for time series
split_idx = int(len(X) * 0.80)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
dates_test = dates[split_idx:]

# Scale features (important for Linear Regression)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training samples : {len(X_train):>4} ({X_train[0] and dates[0].date()} → {dates[split_idx-1].date()})")
print(f"Test samples     : {len(X_test):>4} ({dates[split_idx].date()} → {dates[-1].date()})")

## 6. Model Training

In [ ]:
# ── Linear Regression ─────────────────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
print("✅ Linear Regression trained.")

# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_sc, y_train)
y_pred_rf = rf.predict(X_test_sc)
print("✅ Random Forest trained.")

## 7. Model Evaluation

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R²': r2, 'MAPE (%)': mape}

results = pd.DataFrame([
    evaluate('Linear Regression', y_test, y_pred_lr),
    evaluate('Random Forest',     y_test, y_pred_rf),
])

print("\n📊 Model Evaluation Metrics")
print("=" * 60)
print(results.to_string(index=False, float_format='{:.4f}'.format))
print("\nMetric Guide:")
print("  MAE    — average dollar error (lower = better)")
print("  RMSE   — penalises large errors more (lower = better)")
print("  R²     — variance explained: 1.0 = perfect (higher = better)")
print("  MAPE   — average % error (lower = better)")

## 8. Visualizations

In [ ]:
# ── Actual vs Predicted — Both Models ─────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

for ax, preds, name, color in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Linear Regression', 'Random Forest'],
    [COLORS['linear'], COLORS['rf']]
):
    ax.plot(dates_test, y_test,  color=COLORS['actual'], linewidth=1.8,
            label='Actual Close', zorder=3)
    ax.plot(dates_test, preds,   color=color, linewidth=1.4,
            linestyle='--', label=f'Predicted ({name})', alpha=0.9, zorder=2)
    ax.fill_between(dates_test, y_test, preds, alpha=0.12, color=color)

    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    ax.set_title(f'{TICKER} — {name} | MAE: ${mae:.2f} | R²: {r2:.4f}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Close Price (USD)')
    ax.legend(loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.suptitle(f'{TICKER} Next-Day Close Price Prediction (Test Set)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('predictions_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Residuals Plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, name, color in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Linear Regression', 'Random Forest'],
    [COLORS['linear'], COLORS['rf']]
):
    residuals = y_test - preds
    ax.scatter(preds, residuals, alpha=0.5, color=color, s=18, edgecolors='white', linewidths=0.3)
    ax.axhline(0, color='black', linewidth=1.5, linestyle='--')
    ax.set_xlabel('Predicted Price (USD)')
    ax.set_ylabel('Residual (Actual − Predicted)')
    ax.set_title(f'Residuals — {name}', fontweight='bold')

plt.suptitle('Residual Analysis (closer to 0 = better)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Importance (Random Forest) ────────────────────────────────────────
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(importances.index, importances.values,
               color=plt.cm.viridis(importances.values / importances.max()),
               edgecolor='white')

# Label bars
for bar, val in zip(bars, importances.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('Importance Score')
ax.set_title(f'{TICKER} — Random Forest Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Metrics Comparison Bar Chart ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

metrics  = ['MAE', 'RMSE', 'MAPE (%)']
models   = results['Model'].tolist()
bar_clrs = [COLORS['linear'], COLORS['rf']]

for ax, metric in zip(axes, metrics):
    vals = results[metric].tolist()
    bars = ax.bar(models, vals, color=bar_clrs, edgecolor='white', width=0.5)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + max(vals)*0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(metric, fontweight='bold')
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(['Lin. Reg.', 'Rand. Forest'], fontsize=9)

plt.suptitle('Model Performance Comparison (lower is better)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Results & Key Insights

In [ ]:
best = results.loc[results['MAE'].idxmin(), 'Model']
lr_r = results[results['Model'] == 'Linear Regression'].iloc[0]
rf_r = results[results['Model'] == 'Random Forest'].iloc[0]

print(f"""\n{'='*60}
RESULTS SUMMARY — {TICKER} Next-Day Close Prediction
{'='*60}

Linear Regression:
  MAE    : ${lr_r['MAE']:.2f}   (avg dollar error per prediction)
  RMSE   : ${lr_r['RMSE']:.2f}
  R²     : {lr_r['R²']:.4f}
  MAPE   : {lr_r['MAPE (%)']:.2f}%

Random Forest:
  MAE    : ${rf_r['MAE']:.2f}
  RMSE   : ${rf_r['RMSE']:.2f}
  R²     : {rf_r['R²']:.4f}
  MAPE   : {rf_r['MAPE (%)']:.2f}%

🏆 Best model by MAE: {best}
{'='*60}

KEY INSIGHTS:
1. Lag_1 (yesterday's close) is the single most important feature
   — stock prices are highly auto-correlated day to day.

2. Moving averages (MA_5, MA_10) add meaningful signal beyond
   just raw price levels.

3. Random Forest outperforms Linear Regression because it can
   capture non-linear relationships between features.

4. Despite good R², stock prediction is inherently noisy — these
   models are useful for directional signals, not precise prices.

LIMITATIONS & NEXT STEPS:
- Add technical indicators (RSI, MACD, Bollinger Bands)
- Incorporate sentiment data (news, social media)
- Try LSTM/GRU models for sequential pattern learning
- Use walk-forward validation for more realistic evaluation
""")